In [9]:
import pandas as pd

raw = '../data/raw/'
clean = '../data/clean/'

orders   = pd.read_csv(clean + 'orders_clean.csv', parse_dates=['order_purchase_timestamp'])
reviews  = pd.read_csv(clean + 'reviews_clean.csv')
products = pd.read_csv(clean + 'products_clean.csv')

items     = pd.read_csv(raw + 'olist_order_items_dataset.csv')
customers = pd.read_csv(raw + 'olist_customers_dataset.csv')

print('orders:', len(orders), ' items:', len(items))
print('products unique:', products['product_id'].is_unique)
print('reviews unique:', reviews['order_id'].is_unique)
print('customers unique:', customers['customer_id'].is_unique)

orders: 96203  items: 112650
products unique: True
reviews unique: True
customers unique: True


In [10]:
expected = items['order_id'].isin(orders['order_id']).sum()
print('Expected rows in master:', expected)

Expected rows in master: 109872


In [11]:
order_cols = ['order_id', 'customer_id', 'order_purchase_timestamp',
              'order_month', 'delivery_days', 'is_late']

master = items.merge(orders[order_cols], on='order_id', how='inner')
print('after orders:', len(master))

master = master.merge(
    customers[['customer_id', 'customer_unique_id', 'customer_state', 'customer_city']],
    on='customer_id', how='left')
print('after customers:', len(master))

master = master.merge(products[['product_id', 'category_en']],
                      on='product_id', how='left')
print('after products:', len(master))

master = master.merge(reviews[['order_id', 'review_score']],
                      on='order_id', how='left')
print('after reviews:', len(master))

after orders: 109872
after customers: 109872
after products: 109872
after reviews: 109872


In [12]:
master['category_en'] = master['category_en'].fillna('unknown')

print('Rows match expected:', len(master) == expected)
print(master.isna().sum()[master.isna().sum() > 0])

Rows match expected: True
review_score    823
dtype: int64


In [13]:
items_total = items.loc[items['order_id'].isin(orders['order_id']), 'price'].sum()
master_total = master['price'].sum()

print('Items total :', round(items_total, 2))
print('Master total:', round(master_total, 2))
print('Match:', round(items_total, 2) == round(master_total, 2))

Items total : 13179777.95
Master total: 13179777.95
Match: True


In [17]:
monthly = (master.groupby('order_month')
                 .agg(orders=('order_id', 'nunique'),
                      revenue=('price', 'sum')))
monthly['aov'] = monthly['revenue'] / monthly['orders']
monthly['mom_pct'] = (monthly['revenue'].pct_change() * 100).round(1)
monthly.round(1)

,orders,revenue,aov,mom_pct
order_month,,,,
2017-01,750,111798.4,149.1,NaN
2017-02,1653,234223.4,141.7,109.5
2017-03,2546,359198.8,141.1,53.4
2017-04,2303,340669.7,147.9,-5.2
2017-05,3545,489159.2,138.0,43.6
2017-06,3135,421923.4,134.6,-13.7
2017-07,3872,481604.5,124.4,14.1
2017-08,4193,554699.7,132.3,15.2
2017-09,4150,607399.7,146.4,9.5


In [15]:
master.groupby('category_en')['price'].sum().sort_values(ascending=False).head(10)

category_en
health_beauty            1229557.50
watches_gifts            1163187.91
bed_bath_table           1022955.77
sports_leisure            952661.40
computers_accessories     887944.60
furniture_decor           706237.17
housewares                614341.62
cool_stuff                609158.00
auto                      577721.09
garden_tools              469135.40
Name: price, dtype: float64

In [16]:
master.to_csv('../data/clean/master.csv', index=False)
print('Saved', master.shape)


Saved (109872, 17)
